In [1]:
import numpy as np
from matplotlib import pyplot
import scipy

In [8]:
from google.colab import files
uploaded = files.upload()

Saving ex3weights.mat to ex3weights.mat


In [31]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def forward_prop(Theta1, Theta2, X):
    m = X.shape[0]
    a1 = np.hstack([np.ones([m, 1]), X])
    z2 = np.matmul(a1, Theta1.T)
    a2 = np.hstack([np.ones([m, 1]), sigmoid(z2)])
    z3 = np.matmul(a2, Theta2.T)
    a3 = sigmoid(z3)

    return a1,a2,a3

In [39]:
def cost(theta1, theta2, X, y):
    """
    Compute cost for 2-layer neural network.

    Parameters
    ----------
    theta1 : array_like
        Weights for the first layer in the neural network.
        It has shape (2nd hidden layer size x input size + 1)

    theta2: array_like
        Weights for the second layer in the neural network.
        It has shape (output layer size x 2nd hidden layer size + 1)

    X : array_like
        The inputs having shape (number of examples x number of dimensions).

    y : array_like
        1-hot encoding of labels for the input, having shape
        (number of examples x number of labels).

    lambda_ : float
        The regularization parameter.

    Returns
    -------
    J : float
        The computed value for the cost function.

    """
    a1, a2, h = forward_prop(theta1,theta2, X)

    term1 = y * np.log(h)
    term2 = (1 - y) * np.log(1 - h)

    J = (- 1 / (len(y))) * np.sum(term1 + term2)

    return J

In [33]:
def reg_cost(theta1, theta2, X, y, lambda_):
    """
    Compute cost for 2-layer neural network.

    Parameters
    ----------
    theta1 : array_like
        Weights for the first layer in the neural network.
        It has shape (2nd hidden layer size x input size + 1)

    theta2: array_like
        Weights for the second layer in the neural network.
        It has shape (output layer size x 2nd hidden layer size + 1)

    X : array_like
        The inputs having shape (number of examples x number of dimensions).

    y : array_like
        1-hot encoding of labels for the input, having shape
        (number of examples x number of labels).

    lambda_ : float
        The regularization parameter.

    Returns
    -------
    J : float
        The computed value for the cost function.

    """

    J = cost(theta1, theta2, X, y)
    term3 = np.sum(np.square(theta1[:,1:])) + np.sum(np.square(theta2[:,1:]))

    J += (lambda_ / (2 * len(y))) * term3

    return J

In [47]:
data = scipy.io.loadmat('ex3data1.mat', squeeze_me=True)
y = data['y']
X = data['X']
y_onehot = np.eye(10)[y]


In [48]:
wei = scipy.io.loadmat('ex3weights.mat', squeeze_me=True)
theta1 = wei['Theta1']
theta2 = wei['Theta2']

In [49]:
##coste sin regularizar
coste = cost(theta1, theta2, X, y_onehot)
print(coste)

0.2876291651613189


In [50]:
##coste regularizado
coste_reg = reg_cost(theta1, theta2, X, y_onehot, 1)
print(coste_reg)

0.38376985909092365


In [51]:
def backprop(theta1, theta2, X, y, lambda_):
    """
    Compute cost and gradient for 2-layer neural network.

    Parameters
    ----------
    theta1 : array_like
        Weights for the first layer in the neural network.
        It has shape (2nd hidden layer size x input size + 1)

    theta2: array_like
        Weights for the second layer in the neural network.
        It has shape (output layer size x 2nd hidden layer size + 1)

    X : array_like
        The inputs having shape (number of examples x number of dimensions).

    y : array_like
        1-hot encoding of labels for the input, having shape
        (number of examples x number of labels).

    lambda_ : float
        The regularization parameter.

    Returns
    -------
    J : float
        The computed value for the cost function.

    grad1 : array_like
        Gradient of the cost function with respect to weights
        for the first layer in the neural network, theta1.
        It has shape (2nd hidden layer size x input size + 1)

    grad2 : array_like
        Gradient of the cost function with respect to weights
        for the second layer in the neural network, theta2.
        It has shape (output layer size x 2nd hidden layer size + 1)

    """
    m = len(y)

    Delta1 = np.zeros(np.shape(theta1))
    Delta2 = np.zeros(np.shape(theta2))

    a1, a2, H = forward_prop(theta1, theta2, X)

    d3 = H - y
    d2 = (np.dot(d3,theta2) * (a2 * (1 - a2)))[:,1:]
    Delta1 = np.dot(d2.T, a1)
    Delta2 = np.dot(d3.T, a2)

    Delta1 = Delta1/m
    Delta2 = Delta2/m
    Delta1[:,1:] = Delta1[:,1:] + ( lambda_/m)*theta1[:,1:]
    Delta2[:,1:] = Delta2[:,1:] + ( lambda_/m)*theta2[:,1:]

    J = reg_cost(theta1, theta2, X, y,  lambda_)
    grad1 = Delta1
    grad2 = Delta2

    return (J, grad1, grad2)



In [53]:
def suc(result):
    predicciones = []
    for i in range(len(result)):
        predicciones.append(np.argmax(result[i])+1)

    aciertos = 0
    for i in range(len(y)):
        if (predicciones[i] == y[i]):
            aciertos +=1

    return (aciertos/len(y)*100)

In [62]:
##generamos valores aleatorios
eInit = 0.12
Theta1 = np.random.random((theta1.shape[0],(theta1.shape[1] - 1 + 1)))*(2*eInit) - eInit
Theta2 = np.random.random((10,(theta1.shape[0] + 1)))*(2*eInit) - eInit
thetas =  np.concatenate((np.ravel(Theta1), np.ravel(Theta2)))

lambda_values = [0.1, 0.3, 0.5, 0.8, 1]
results = []
for lambda_ in lambda_values:
  min = scipy.optimize.minimize(fun=backprop, x0=Theta1.ravel(), args=(Theta2, X, y_onehot, lambda_values), method='TNC', jac=True, options={'maxiter': 1000})
  results.append(suc(min))

<ipython-input-62-fdf6cd90d023>:10: OptimizeWarning: Unknown solver options: maxiter
  min = scipy.optimize.minimize(fun=backprop, x0=Theta1.ravel(), args=(Theta2, X, y_onehot, lambda_values), method='TNC', jac=True, options={'maxiter': 1000})


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10025 is different from 401)

In [68]:


def train_neural_network(X, y, input_size, hidden_size, num_labels, lambda_, alpha, max_iter):
    epsilon_init = 0.12
    theta1 = np.random.rand(hidden_size, input_size + 1) * 2 * epsilon_init - epsilon_init
    theta2 = np.random.rand(num_labels, hidden_size + 1) * 2 * epsilon_init - epsilon_init

    m = len(y)

    accuracies = []
    for l in lambda_:
        for _ in range(max_iter):
            J, grad1, grad2 = backprop(theta1, theta2, X, y, l)

            theta1 -= alpha * grad1
            theta2 -= alpha * grad2

        _, _, H = forward_prop(theta1, theta2, X)
        predictions = np.argmax(H, axis=1)
        accuracy = np.mean(predictions == np.argmax(y, axis=1)) * 100
        accuracies.append(accuracy)

    return accuracies



In [69]:
##Entrenamos sin usar scipy.optimize.minimize

input_size = theta1.shape[0]
hidden_size = theta1.shape[1]
num_labels = 10
alpha = 1.0
max_iter = 1000
lambda_values = [0.1, 0.3, 0.5, 0.8, 1]


# Train neural network for different lambda values
accuracies = train_neural_network(X, y_onehot, input_size, hidden_size, num_labels, lambda_values, alpha, max_iter)

# Plot accuracy vs lambda
plt.plot(lambda_values, accuracies, marker='o')
plt.title('Accuracy vs Lambda')
plt.xlabel('Lambda')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.show()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 26 is different from 401)